# Recent Wildfire Detections: A Reproducible Interactive Mapping Workflow

# Project 2: Wildfire Mapping
### SDS210 — Programming with Spatial Data
**University of Zurich · FS 2026**

---

## Project Overview

This project is organised as a pipeline of four notebooks. Each notebook produces an intermediate file that the next one consumes, so they must be executed **in the following order**:

1. **`data_retrieval.ipynb`** — Retrieves live wildfire detection data from the NASA FIRMS API (VIIRS S-NPP, Near Real-Time) and caches it locally as a CSV.
2. **`data_cleaning.ipynb`** — Inspects and cleans the raw records — handling missing values, formatting dates and times, and classifying confidence levels — and writes a tidy dataset to `data/processed/`.
3. **`exploratory_analysis.ipynb`** — Analyses four targeted spatial and statistical questions about global fire activity using the cleaned dataset.
4. **`interactive_web_map.ipynb`** — Visualises the results on a fully interactive Folium web map with graduated symbology, exported to `outputs/`.

> **Run order matters:** each notebook reads the output of the previous step. Running them out of order (or skipping one) will cause `FileNotFoundError` or stale results. Restart the kernel and run all cells top-to-bottom in each notebook before moving on to the next.

### Research Questions

| # | Question |
|---|----------|
| Q1 | Where are the most severe active wildfires (highest Fire Radiative Power)? |
| Q2 | How does thermal intensity (FRP) vary across fire events and continents? |
| Q3 | Are wildfires more commonly detected during daytime or nighttime passes, and does this differ by region? |
| Q4 | How does detection confidence relate to fire intensity (FRP)? |

### Data Source

**NASA FIRMS** (Fire Information for Resource Management System)  
Product: **VIIRS S-NPP NRT** (Near Real-Time, 375 m resolution)  
API: https://firms.modaps.eosdis.nasa.gov/api/  
Time window: last **7 days**, global extent

> **API Key Required:** Register for a free MAP_KEY at https://firms.modaps.eosdis.nasa.gov/api/map_key/.  
> Enter your key in the configuration cell below. If no key is available, the notebook will automatically fall back to a locally cached CSV file in `data/raw/`.

---
## 1. Setup

All imports are grouped here in a single cell. This cell must run first.

In [3]:
pip install folium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [folium]
Note: you may need to restart the kernel to use updated packages.


In [4]:
# --- Standard library ---
import os
import warnings
from pathlib import Path
from datetime import datetime, timezone

# --- Data manipulation ---
import numpy as np
import pandas as pd

# --- Geospatial ---
import geopandas as gpd
from shapely.geometry import Point

# --- Visualisation ---
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import folium
from folium.plugins import MarkerCluster, MiniMap, Fullscreen

# --- HTTP requests (for API access) ---
import requests
from io import StringIO

# Suppress minor deprecation warnings for a clean output
warnings.filterwarnings("ignore", category=FutureWarning)

# Set a consistent visual style for all static plots
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

print(f"Setup complete. Timestamp: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}")

Setup complete. Timestamp: 2026-05-11 18:43 UTC


---
## 2. Data Access — NASA FIRMS API

### 2.1 Configuration

To successfully access the API I entered my free NASA FIRMS MAP_KEY below.
Note: The MAP KEY is valid for both FIRMS (Global) and FIRMS (US/Canada) sites.
Transaction limit: 5000 transactions / 10 minutes (view status: https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?MAP_KEY=8243f7094a0fcb92281c5c9033b00ff2) 
If `MAP_KEY` is left as an empty string `""`, the notebook falls back to a local CSV file.

In [18]:
# ============================================================
#  USER CONFIGURATION
# ============================================================

MAP_KEY = "8243f7094a0fcb92281c5c9033b00ff2"          # <-- NASA FIRMS MAP_KEY (required for API access; obtain from https://firms.modaps.eosdis.nasa.gov/api/)
DAYS    = 5           # Number of days of data to retrieve (1–10)
CACHE_FILE = DATA_DIR / "firms_viirs_global_5d.csv"
SOURCE  = "VIIRS_SNPP_NRT"   # FIRMS product identifier
AREA = "world"                  # or bbox: "5.9,45.8,10.5,47.9" for Switzerland approx.

# Relative paths (no absolute paths — ensures reproducibility on any machine)
PROJECT_ROOT = Path("..").resolve()
DATA_DIR    = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR  = PROJECT_ROOT / "outputs"
CACHE_FILE  = DATA_DIR / "firms_viirs_global_5d.csv"
MAP_OUTPUT  = OUTPUT_DIR / "wildfire_map.html"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root    : {PROJECT_ROOT}")
print(f"Data directory  : {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Cache file      : {CACHE_FILE}")
print(f"Map output      : {MAP_OUTPUT}")

Project root    : /Users/davidleu/Desktop/sds210-wildfire-mapping-project
Data directory  : /Users/davidleu/Desktop/sds210-wildfire-mapping-project/data/raw
Output directory: /Users/davidleu/Desktop/sds210-wildfire-mapping-project/outputs
Cache file      : /Users/davidleu/Desktop/sds210-wildfire-mapping-project/data/raw/firms_viirs_global_5d.csv
Map output      : /Users/davidleu/Desktop/sds210-wildfire-mapping-project/outputs/wildfire_map.html


### Reproducibility note
This project uses relative paths so that another student or instructor can run the notebook on a different machine without changing absolute file paths.

### 2.2 Fetch Function

A reusable function handles both the live API call and the local fallback. This keeps the retrieval logic in one place (DRY principle) and makes the data source completely transparent.

In [19]:
def fetch_firms_data(map_key: str, source: str, days: int, cache_path: Path) -> pd.DataFrame:
    """
    Retrieve wildfire detection records from the NASA FIRMS API.

    Attempts a live API download when a valid map_key is supplied.
    Falls back to a cached local CSV at cache_path when the key is
    empty or the request fails.

    Parameters
    ----------
    map_key   : NASA FIRMS MAP_KEY string (empty → local fallback).
    source    : FIRMS product name, e.g. 'VIIRS_SNPP_NRT'.
    days      : Number of days of data to request (1–10).
    cache_path: Path to the local CSV fallback file.

    Returns
    -------
    pd.DataFrame with raw FIRMS records.

    Raises
    ------
    FileNotFoundError if no key is given and no local cache exists.
    """
    # --- Live API path ---
    if map_key.strip():
        base_url = "https://firms.modaps.eosdis.nasa.gov/api/area/csv"
        area     = "world"   # full global extent
        url = f"{base_url}/{MAP_KEY}/{SOURCE}/{AREA}/{DAYS}"

        print(f"Connecting to FIRMS API … ({source}, {days} days, global)")
        try:
            response = requests.get(url, timeout=60)
            response.raise_for_status()          # raises HTTPError for 4xx / 5xx

            raw_df = pd.read_csv(StringIO(response.text))

            # Persist a local copy so the notebook can work offline next run
            raw_df.to_csv(cache_path, index=False)
            print(f"Success — {len(raw_df):,} records retrieved. Cached to {cache_path}")
            return raw_df

        except requests.exceptions.RequestException as exc:
            print(f"API request failed ({exc}). Falling back to local cache …")

    else:
        print("No MAP_KEY provided — using local cache.")

    # --- Local fallback path ---
    if cache_path.exists():
        raw_df = pd.read_csv(cache_path)
        print(f"Loaded {len(raw_df):,} records from {cache_path}")
        return raw_df

    raise FileNotFoundError(
        f"No API key was given and no local cache was found at '{cache_path}'.\n"
        "Please either:\n"
        "  1) Register for a free MAP_KEY at https://firms.modaps.eosdis.nasa.gov/api/map_key/\n"
        "  2) Manually download a CSV from FIRMS and place it at the path above."
    )

In [20]:
print("DAYS =", DAYS)
print("SOURCE =", SOURCE)
print("CACHE_FILE =", CACHE_FILE)

DAYS = 5
SOURCE = VIIRS_SNPP_NRT
CACHE_FILE = /Users/davidleu/Desktop/sds210-wildfire-mapping-project/data/raw/firms_viirs_global_5d.csv


In [21]:
# --- Execute the data retrieval ---
raw_df = fetch_firms_data(
    map_key    = MAP_KEY,
    source     = SOURCE,
    days       = DAYS,
    cache_path = CACHE_FILE
)

Connecting to FIRMS API … (VIIRS_SNPP_NRT, 5 days, global)
Success — 133,061 records retrieved. Cached to /Users/davidleu/Desktop/sds210-wildfire-mapping-project/data/raw/firms_viirs_global_5d.csv


---

## Reproducibility notes

To make this project reproducible:

- keep all data in the `data/` folder
- save final outputs to `outputs/`
- use only relative paths
- document package requirements
- ensure the notebook runs from top to bottom without manual intervention